# Hands-on 1: train a model and track it with MLflow

We train a model that tells **active galaxies** (quasars, blazars, AGN) from **stars**, using real ZTF alerts. MLflow keeps a record of every attempt.

*On entraîne un modèle qui distingue les galaxies actives des étoiles, à partir de vraies alertes ZTF. MLflow garde une trace de chaque essai.*

## 1. Start MLflow

In a terminal, in this folder:

```bash
mlflow server --host 127.0.0.1 --port 6969
```

Leave it running and open http://127.0.0.1:6969

*Dans un terminal, lancez la commande, laissez-la tourner et ouvrez l'adresse dans le navigateur.*

**You should see / Vous devriez voir :** the MLflow home page. Click **Model training** at the top left, then **Experiments**.

![the MLflow home page. Click **Model training** at the top left, then **Experiments**.](images/01_mlflow_home.png)

In [ ]:
import logging
import mlflow

logging.getLogger("mlflow").setLevel(logging.ERROR)  # hide MLflow info messages
mlflow.set_tracking_uri("http://127.0.0.1:6969")
mlflow.set_experiment("fink-td")
print("connected to MLflow")

## 2. Look at an alert

`data/alerts.parquet` holds 320 ZTF alerts processed by Fink. The `cdsxmatch` column is the SIMBAD class found by Fink.

*Le fichier contient 320 alertes ZTF traitées par Fink. La colonne `cdsxmatch` est la classe SIMBAD trouvée par Fink.*

In [ ]:
import pandas as pd
import preprocessing as pp

alerts = pd.read_parquet("data/alerts.parquet")
alerts[["objectId", "cdsxmatch"]].head()

In [ ]:
import matplotlib.pyplot as plt

alert = alerts.iloc[0]
lc = pp.light_curve(alert)

days = lc["jd"] - lc["jd"].min()
plt.scatter(days, lc["mag"], c=lc["band"], cmap="RdYlGn_r")
plt.gca().invert_yaxis()
plt.xlabel("Days since the first point")
plt.ylabel("Magnitude")
plt.title(alert["cdsxmatch"])
plt.show()

**You should see / Vous devriez voir :** a light curve. Each point is one measurement; green is the g band, red is the r band. Bright is up.

![a light curve. Each point is one measurement; green is the g band, red is the r band. Bright is up.](images/02_light_curve.png)

## 3. Filter, features and labels

Open `preprocessing.py`. It has three functions:

- `make_cut`: the **filter**, which alerts we keep;
- `make_X`: the **features**, the numbers the model sees;
- `make_labels`: the **answer**, 1 for an active galaxy and 0 for a star.

*`make_cut` est le filtre, `make_X` calcule les features, `make_labels` donne la réponse.*

In [ ]:
kept = pp.make_cut(alerts)
y = pp.make_labels(kept)

# keep only the alerts that have an answer
kept = kept[y.notna()]
y = y[y.notna()].astype(int)

X = pp.make_X(kept)
print(len(X), "alerts:", (y == 1).sum(), "active galaxies,", (y == 0).sum(), "stars")
X.head()

**You should see / Vous devriez voir :** 135 alerts (30 active galaxies, 105 stars) and a table of 7 features.

![135 alerts (30 active galaxies, 105 stars) and a table of 7 features.](images/03_features.png)

## 4. Train and log a run

`train_test_split` keeps 30% of the alerts aside to test the model. Everything inside `with mlflow.start_run()` is saved in MLflow.

*On garde 30 % des alertes pour tester le modèle. Tout ce qui est dans le bloc `with` est enregistré dans MLflow.*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)


def train(n_trees, max_depth, run_name):
    with mlflow.start_run(run_name=run_name):
        model = RandomForestClassifier(n_estimators=n_trees, max_depth=max_depth, random_state=0)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        mlflow.log_param("n_trees", n_trees)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy_score(y_test, predictions))
        mlflow.log_metric("f1", f1_score(y_test, predictions))
        mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")
        mlflow.log_artifact("preprocessing.py")
        mlflow.log_artifact("processor.py")

    print(run_name, "f1 =", round(f1_score(y_test, predictions), 2))
    return model


model = train(n_trees=100, max_depth=3, run_name="first_try")

Open the MLflow page, click on the `fink-td` experiment, then on `first_try`.

*Dans MLflow, ouvrez l'expérience `fink-td`, puis le run `first_try`.*

**You should see / Vous devriez voir :** the run page, with its parameters, its metrics, the model and the code files.

![the run page, with its parameters, its metrics, the model and the code files.](images/04_run_page.png)

## 5. Try several settings

In [ ]:
for depth in [2, 5, 10]:
    train(n_trees=100, max_depth=depth, run_name=f"depth_{depth}")

**You should see / Vous devriez voir :** four runs in the `fink-td` experiment.

![four runs in the `fink-td` experiment.](images/05_runs_list.png)

Tick the runs and click **Compare**.

*Cochez les runs et cliquez sur **Compare**.*

**You should see / Vous devriez voir :** the four runs ticked, and the **Compare** button.

![the four runs ticked, and the **Compare** button.](images/06a_select_compare.png)

**You should see / Vous devriez voir :** the runs side by side. Scroll down to **Metrics** to compare the f1.

![the runs side by side. Scroll down to **Metrics** to compare the f1.](images/06_compare.png)

## Exercises / Exercices

1. In `make_X`, add a feature, for example the number of points: `"n_points": len(lc)`. Re-run sections 3 and 4. Does the f1 go up?
2. In `make_cut`, ask for 5 points instead of 3. How many alerts are left?
3. Change `random_state=0` to `1`, then `2` in `train_test_split`. The f1 changes a lot. Why? (Hint: how many active galaxies are in the test set?)

*1. Ajoutez une feature. 2. Changez le filtre. 3. Changez `random_state` : pourquoi le f1 bouge-t-il autant ?*

In [ ]:
# your tries here / vos essais ici
